# 🧠 QUANTOPS — Phase 2 Autonomous Signal Scanner
**Version:** 2.0 | **Author:** DJ (Howrie) | **Built with:** COORDINATOR + PHANTOM + NEXUS

---
### What this does
- Runs on a schedule (every N minutes via a loop)
- Polls Bybit for candle data on your watchlist
- Applies Howrie Band logic (EMA-based trend filter)
- Scores each coin across 5 signal conditions
- Fires a Telegram alert when a coin hits threshold
- Logs every scan to a timestamped CSV

### Setup required
1. Add your Telegram Bot Token + Chat ID in Cell 2
2. Optionally add your LunarCrush API key for sentiment scoring
3. Edit your watchlist in Cell 3
4. Run all cells, then run the scanner loop cell

---
> ⚠️ **HOWRIE BAND RULE: Blue = long/hold. Red = IMMEDIATE EXIT. No exceptions.**
> ⚠️ **ARC RULE: Never average down. Ever.**

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
!pip install requests pandas numpy python-telegram-bot==13.15 pybit --quiet
print('✅ Dependencies installed')

In [ ]:
# ============================================================
# CELL 2 — CONFIG  (edit these)
# ============================================================

CONFIG = {
    # --- Telegram ---
    # 1. Message @BotFather on Telegram → /newbot → copy token
    # 2. Message @userinfobot → copy your chat_id
    'TELEGRAM_TOKEN':   'YOUR_BOT_TOKEN_HERE',
    'TELEGRAM_CHAT_ID': 'YOUR_CHAT_ID_HERE',

    # --- LunarCrush (optional — leave blank to skip sentiment) ---
    'LUNARCRUSH_KEY': '',

    # --- Scanner settings ---
    'SCAN_INTERVAL_MINUTES': 15,      # how often to scan
    'SIGNAL_THRESHOLD':      3,       # out of 5 — minimum score to alert
    'CANDLE_INTERVAL':       '15',    # Bybit interval: 1,3,5,15,30,60,120,240,D
    'CANDLE_LIMIT':          100,     # candles to fetch per coin

    # --- Risk gates ---
    'MAX_ALERTS_PER_SCAN':   3,       # don't spam — cap alerts per run
    'COOLDOWN_MINUTES':      60,      # don't re-alert same coin within N mins
}

print('✅ Config loaded')

In [ ]:
# ============================================================
# CELL 3 — WATCHLIST  (edit this)
# ============================================================

WATCHLIST = [
    'BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'AVAXUSDT',
    'TAOUSDT', 'RENDERUSDT', 'HYPEUSDT',
    'NEARUSDT', 'AKTUSDT', 'STRKUSDT',
    'WLDUSDT', 'FILUSDT',
    'PIPPINUSDT', 'SIRENUSDT',
]

# Coins you currently HOLD (skip entry signals for these)
CURRENT_POSITIONS = ['TAOUSDT', 'RENDERUSDT', 'HYPEUSDT']

print(f'✅ Watchlist loaded — {len(WATCHLIST)} coins')

In [ ]:
# ============================================================
# CELL 4 — CORE MODULES
# ============================================================

import requests
import pandas as pd
import numpy as np
import time
import logging
from datetime import datetime, timedelta
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
log = logging.getLogger('QUANTOPS')

# ── Alert cooldown tracker ──────────────────────────────────
last_alerted = {}  # symbol → datetime

# ── Scan log ───────────────────────────────────────────────
scan_log = []


# ═══════════════════════════════════════════════════════════
# BYBIT DATA FEED
# ═══════════════════════════════════════════════════════════

def fetch_candles(symbol: str, interval: str = '15', limit: int = 100) -> pd.DataFrame | None:
    """Fetch OHLCV candles from Bybit REST API."""
    url = 'https://api.bybit.com/v5/market/kline'
    params = {
        'category': 'linear',
        'symbol':   symbol,
        'interval': interval,
        'limit':    limit,
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if data.get('retCode') != 0:
            log.warning(f'{symbol}: Bybit error — {data.get("retMsg")}')
            return None
        rows = data['result']['list']
        df = pd.DataFrame(rows, columns=['ts','open','high','low','close','volume','turnover'])
        df = df.astype({'ts': 'int64','open':'float','high':'float','low':'float',
                        'close':'float','volume':'float','turnover':'float'})
        df['ts'] = pd.to_datetime(df['ts'], unit='ms')
        df = df.sort_values('ts').reset_index(drop=True)
        return df
    except Exception as e:
        log.error(f'{symbol}: fetch_candles error — {e}')
        return None


# ═══════════════════════════════════════════════════════════
# HOWRIE BAND  (EMA-based trend filter)
# Blue  → price above band → LONG / HOLD
# Red   → price below band → IMMEDIATE EXIT
# ═══════════════════════════════════════════════════════════

def calc_howrie_band(df: pd.DataFrame,
                     fast_period: int = 9,
                     slow_period: int = 21,
                     atr_period:  int = 14,
                     band_mult:   float = 1.5) -> dict:
    """
    Howrie Band implementation:
    - Fast EMA + Slow EMA define the mid-trend
    - ATR-based envelope creates upper/lower band
    - Blue = close > upper band of slow EMA → bullish
    - Red  = close < lower band of slow EMA → bearish
    """
    close = df['close']
    high  = df['high']
    low   = df['low']

    # EMAs
    fast_ema = close.ewm(span=fast_period, adjust=False).mean()
    slow_ema = close.ewm(span=slow_period, adjust=False).mean()

    # ATR
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low  - close.shift()).abs()
    ], axis=1).max(axis=1)
    atr = tr.ewm(span=atr_period, adjust=False).mean()

    # Band
    upper_band = slow_ema + band_mult * atr
    lower_band = slow_ema - band_mult * atr

    last_close      = close.iloc[-1]
    last_fast_ema   = fast_ema.iloc[-1]
    last_slow_ema   = slow_ema.iloc[-1]
    last_upper_band = upper_band.iloc[-1]
    last_lower_band = lower_band.iloc[-1]
    last_atr        = atr.iloc[-1]

    # Colour state
    if last_close > last_upper_band and last_fast_ema > last_slow_ema:
        band_color = 'BLUE'
    elif last_close < last_lower_band or last_fast_ema < last_slow_ema:
        band_color = 'RED'
    else:
        band_color = 'NEUTRAL'

    # Band squeeze (low ATR relative to price = compression)
    band_width_pct = ((last_upper_band - last_lower_band) / last_slow_ema) * 100

    return {
        'color':       band_color,
        'fast_ema':    round(last_fast_ema, 6),
        'slow_ema':    round(last_slow_ema, 6),
        'upper_band':  round(last_upper_band, 6),
        'lower_band':  round(last_lower_band, 6),
        'atr':         round(last_atr, 6),
        'band_width':  round(band_width_pct, 2),
        'close':       round(last_close, 6),
    }


# ═══════════════════════════════════════════════════════════
# SIGNAL SCORING ENGINE  (5 conditions → score 0–5)
# ═══════════════════════════════════════════════════════════

def score_coin(df: pd.DataFrame, band: dict) -> dict:
    """
    5-condition SWWA-inspired scoring:
    1. Howrie Band BLUE
    2. Volume spike  (current vol > 1.5× 20-bar avg)
    3. RSI alignment (RSI 45–70 = bullish momentum zone)
    4. Trend compression breakout (band width tightening then expanding)
    5. Price above 50-bar EMA
    """
    conditions = {}
    close  = df['close']
    volume = df['volume']

    # Condition 1 — Howrie Band BLUE
    conditions['howrie_blue'] = band['color'] == 'BLUE'

    # Condition 2 — Volume spike
    vol_avg = volume.iloc[-21:-1].mean()
    vol_current = volume.iloc[-1]
    conditions['volume_spike'] = vol_current > (vol_avg * 1.5) if vol_avg > 0 else False

    # Condition 3 — RSI alignment (14-period)
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
    rs    = gain / loss.replace(0, np.nan)
    rsi   = (100 - 100 / (1 + rs)).iloc[-1]
    conditions['rsi_aligned'] = 45 <= rsi <= 72

    # Condition 4 — Band compression breakout
    # Band was tight N bars ago, now expanding → breakout signal
    if len(df) >= 20:
        past_band = calc_howrie_band(df.iloc[:-5])   # 5 bars ago
        was_tight = past_band['band_width'] < band['band_width']
        conditions['compression_breakout'] = was_tight and band['color'] == 'BLUE'
    else:
        conditions['compression_breakout'] = False

    # Condition 5 — Price above 50 EMA
    ema50 = close.ewm(span=50, adjust=False).mean().iloc[-1]
    conditions['above_ema50'] = close.iloc[-1] > ema50

    score = sum(conditions.values())
    return {
        'score':      score,
        'conditions': conditions,
        'rsi':        round(rsi, 1),
        'vol_ratio':  round(vol_current / vol_avg, 2) if vol_avg > 0 else 0,
    }


# ═══════════════════════════════════════════════════════════
# LUNARCRUSH SENTIMENT  (optional)
# ═══════════════════════════════════════════════════════════

def fetch_sentiment(symbol: str) -> dict:
    """Fetch galaxy score and sentiment from LunarCrush."""
    if not CONFIG['LUNARCRUSH_KEY']:
        return {'galaxy_score': None, 'sentiment': None}
    ticker = symbol.replace('USDT','').lower()
    url    = f'https://lunarcrush.com/api4/public/coins/{ticker}/v1'
    headers = {'Authorization': f'Bearer {CONFIG["LUNARCRUSH_KEY"]}'}
    try:
        r = requests.get(url, headers=headers, timeout=8)
        d = r.json().get('data', {})
        return {
            'galaxy_score': d.get('galaxy_score'),
            'sentiment':    d.get('sentiment'),
            'alt_rank':     d.get('alt_rank'),
        }
    except:
        return {'galaxy_score': None, 'sentiment': None}


# ═══════════════════════════════════════════════════════════
# TELEGRAM ALERTS
# ═══════════════════════════════════════════════════════════

def send_telegram(message: str) -> bool:
    """Send a Telegram message via Bot API."""
    if CONFIG['TELEGRAM_TOKEN'] == 'YOUR_BOT_TOKEN_HERE':
        print(f'[TELEGRAM MOCK]\n{message}')
        return True
    url = f'https://api.telegram.org/bot{CONFIG["TELEGRAM_TOKEN"]}/sendMessage'
    payload = {
        'chat_id':    CONFIG['TELEGRAM_CHAT_ID'],
        'text':       message,
        'parse_mode': 'HTML',
    }
    try:
        r = requests.post(url, json=payload, timeout=10)
        return r.status_code == 200
    except Exception as e:
        log.error(f'Telegram send failed: {e}')
        return False


def format_alert(symbol: str, band: dict, score_data: dict, sentiment: dict) -> str:
    """Format a clean Telegram alert message."""
    cond = score_data['conditions']
    checks = {
        'Howrie Band BLUE':     cond['howrie_blue'],
        'Volume spike':         cond['volume_spike'],
        'RSI aligned':          cond['rsi_aligned'],
        'Compression breakout': cond['compression_breakout'],
        'Above EMA50':          cond['above_ema50'],
    }
    check_lines = '\n'.join(
        f"  {'✅' if v else '❌'} {k}" for k, v in checks.items()
    )
    galaxy = f"  🌙 Galaxy Score: {sentiment['galaxy_score']}" if sentiment.get('galaxy_score') else ''

    return (
        f"🚨 <b>QUANTOPS SIGNAL</b>\n"
        f"━━━━━━━━━━━━━━━━━━━━\n"
        f"<b>{symbol}</b> | Score: {score_data['score']}/5 | {band['color']}\n"
        f"Price: ${band['close']:,.4f} | RSI: {score_data['rsi']} | Vol×{score_data['vol_ratio']}\n\n"
        f"<b>Conditions:</b>\n{check_lines}\n"
        f"{galaxy}\n"
        f"━━━━━━━━━━━━━━━━━━━━\n"
        f"⏰ {datetime.now().strftime('%H:%M:%S AWST')}\n"
        f"⚠️ Howrie Band RED = EXIT IMMEDIATELY"
    )


# ═══════════════════════════════════════════════════════════
# EXIT ALERT  (for current positions when band turns RED)
# ═══════════════════════════════════════════════════════════

def check_exit_signals(positions: list) -> None:
    """Monitor open positions and alert immediately if Howrie Band turns RED."""
    for symbol in positions:
        df = fetch_candles(symbol, CONFIG['CANDLE_INTERVAL'], CONFIG['CANDLE_LIMIT'])
        if df is None:
            continue
        band = calc_howrie_band(df)
        if band['color'] == 'RED':
            msg = (
                f"🔴 <b>EXIT SIGNAL — {symbol}</b>\n"
                f"━━━━━━━━━━━━━━━━━━━━\n"
                f"Howrie Band has turned RED\n"
                f"Price: ${band['close']:,.4f}\n"
                f"<b>RULE: IMMEDIATE EXIT. NO EXCEPTIONS.</b>\n"
                f"⏰ {datetime.now().strftime('%H:%M:%S')}"
            )
            send_telegram(msg)
            log.warning(f'🔴 EXIT SIGNAL: {symbol}')


print('✅ Core modules loaded')

In [ ]:
# ============================================================
# CELL 5 — SINGLE SCAN FUNCTION
# ============================================================

def run_scan(watchlist: list, positions: list, verbose: bool = True) -> pd.DataFrame:
    """
    Run one full scan across the watchlist.
    Returns a scored DataFrame sorted by signal strength.
    """
    log.info(f'--- SCAN START | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} ---')

    # Step 1: Check exit signals on open positions first
    if positions:
        log.info(f'Checking exit signals on {len(positions)} open positions...')
        check_exit_signals(positions)

    results = []
    alerts_fired = 0

    for symbol in watchlist:
        try:
            # Fetch data
            df = fetch_candles(symbol, CONFIG['CANDLE_INTERVAL'], CONFIG['CANDLE_LIMIT'])
            if df is None or len(df) < 55:
                continue

            # Calculate band + score
            band       = calc_howrie_band(df)
            score_data = score_coin(df, band)
            sentiment  = fetch_sentiment(symbol)

            row = {
                'symbol':       symbol,
                'price':        band['close'],
                'band':         band['color'],
                'score':        score_data['score'],
                'rsi':          score_data['rsi'],
                'vol_ratio':    score_data['vol_ratio'],
                'band_width':   band['band_width'],
                'atr':          band['atr'],
                'galaxy_score': sentiment.get('galaxy_score'),
                'in_position':  symbol in positions,
                'scanned_at':   datetime.now(),
            }
            results.append(row)

            # Fire alert if threshold met + not in cooldown + not already in position
            is_new_signal    = score_data['score'] >= CONFIG['SIGNAL_THRESHOLD']
            is_blue          = band['color'] == 'BLUE'
            not_in_position  = symbol not in positions
            cooldown_ok      = (
                symbol not in last_alerted or
                (datetime.now() - last_alerted[symbol]).seconds // 60 >= CONFIG['COOLDOWN_MINUTES']
            )

            if is_new_signal and is_blue and not_in_position and cooldown_ok:
                if alerts_fired < CONFIG['MAX_ALERTS_PER_SCAN']:
                    msg = format_alert(symbol, band, score_data, sentiment)
                    sent = send_telegram(msg)
                    if sent:
                        last_alerted[symbol] = datetime.now()
                        alerts_fired += 1
                        log.info(f'🚨 ALERT SENT: {symbol} | Score {score_data["score"]}/5')

            time.sleep(0.3)  # gentle rate limiting

        except Exception as e:
            log.error(f'{symbol}: scan error — {e}')
            continue

    results_df = pd.DataFrame(results).sort_values('score', ascending=False)

    if verbose and not results_df.empty:
        print('\n' + '═'*65)
        print(f'  QUANTOPS SCAN | {datetime.now().strftime("%H:%M:%S")} | {alerts_fired} alert(s) fired')
        print('═'*65)
        display_cols = ['symbol','price','band','score','rsi','vol_ratio','in_position']
        print(results_df[display_cols].to_string(index=False))
        print('═'*65 + '\n')

    scan_log.append({
        'scan_time':    datetime.now(),
        'coins_scanned': len(results),
        'alerts_fired':  alerts_fired,
    })

    return results_df


print('✅ Scan function ready')

In [ ]:
# ============================================================
# CELL 6 — TEST: Run a single scan now
# ============================================================

print('Running test scan...')
results = run_scan(WATCHLIST, CURRENT_POSITIONS, verbose=True)

# Show top signals
top = results[results['score'] >= 3]
if not top.empty:
    print(f'\n🎯 Top signals (score ≥ 3):')
    print(top[['symbol','band','score','rsi','vol_ratio']].to_string(index=False))
else:
    print('No coins above threshold this scan.')

In [ ]:
# ============================================================
# CELL 7 — AUTONOMOUS LOOP  (runs until you stop it)
#
# ⚡ Run this cell last. It will scan on schedule indefinitely.
# ⛔ Stop it: Runtime → Interrupt Execution  (or Ctrl+C)
# ============================================================

import time as _time

INTERVAL_SECONDS = CONFIG['SCAN_INTERVAL_MINUTES'] * 60
scan_count = 0

print('🤖 QUANTOPS AUTONOMOUS SCANNER STARTED')
print(f'   Interval:  every {CONFIG["SCAN_INTERVAL_MINUTES"]} minutes')
print(f'   Threshold: {CONFIG["SIGNAL_THRESHOLD"]}/5 conditions')
print(f'   Watchlist: {len(WATCHLIST)} coins')
print(f'   Positions: {CURRENT_POSITIONS}')
print('   Stop: Runtime → Interrupt Execution')
print('─'*50)

# Send startup message to Telegram
send_telegram(
    f'🤖 <b>QUANTOPS SCANNER ONLINE</b>\n'
    f'Scanning {len(WATCHLIST)} coins every {CONFIG["SCAN_INTERVAL_MINUTES"]}min\n'
    f'Threshold: {CONFIG["SIGNAL_THRESHOLD"]}/5\n'
    f'Positions monitored: {", ".join(CURRENT_POSITIONS) or "None"}'
)

try:
    while True:
        scan_count += 1
        log.info(f'Scan #{scan_count} starting...')

        results = run_scan(WATCHLIST, CURRENT_POSITIONS, verbose=True)

        # Save results to CSV (appends each scan)
        results['scan_num'] = scan_count
        csv_path = f'quantops_scan_log.csv'
        results.to_csv(csv_path, mode='a', header=(scan_count == 1), index=False)

        next_scan = datetime.now() + timedelta(seconds=INTERVAL_SECONDS)
        log.info(f'Next scan at {next_scan.strftime("%H:%M:%S")} | sleeping {CONFIG["SCAN_INTERVAL_MINUTES"]}min...')
        _time.sleep(INTERVAL_SECONDS)

except KeyboardInterrupt:
    print('\n⛔ Scanner stopped by user.')
    send_telegram('⛔ QUANTOPS scanner stopped.')
    print(f'Total scans completed: {scan_count}')
    print(f'Log saved to: quantops_scan_log.csv')

In [ ]:
# ============================================================
# CELL 8 — REVIEW SCAN LOG  (run anytime)
# ============================================================

try:
    log_df = pd.read_csv('quantops_scan_log.csv')
    print(f'Total rows in log: {len(log_df)}')
    print(f'\nTop signals across all scans:')
    top_all = log_df[log_df['score'] >= 3].sort_values('score', ascending=False)
    print(top_all[['symbol','band','score','rsi','vol_ratio','scanned_at']].head(20).to_string(index=False))
except FileNotFoundError:
    print('No log file yet — run the scanner first.')